In [12]:
import os
import re
import datetime
import pandas as pd
import pdfplumber
from time import sleep
from bs4 import BeautifulSoup
from DrissionPage import ChromiumPage, ChromiumOptions


In [13]:
regulatorName = 'CW CBCSCW'
print(f'Running {regulatorName} Web Scraping Tool v.1.1')

now = datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
os.makedirs(tempfolder, exist_ok=True)
cleanup_tempfolder_after_export = False  # Keep PDFs while tuning parser validation.
registry_pdf_files = [
    os.path.join(tempfolder, file)
    for file in sorted(os.listdir(tempfolder))
    if file.lower().endswith('.pdf') and 'registry_of_supervised_institutions' in file.lower()
]

if registry_pdf_files:
    print(f'[INFO] : found {len(registry_pdf_files)} registry PDF file(s) in tempfolder')


Running CW CBCSCW Web Scraping Tool v.1.1
[INFO] : found 1 registry PDF file(s) in tempfolder


In [14]:
driver = None
if not registry_pdf_files:
    # DrissionPage handles the Cloudflare/browser challenge that blocks plain Selenium.
    options = ChromiumOptions()
    options.set_download_path(tempfolder)
    driver = ChromiumPage(options)
else:
    print('[INFO] : registry PDF detected; browser scraping will be skipped')


[INFO] : registry PDF detected; browser scraping will be skipped


In [15]:
regdict = {regulatorName + ' 1': 'https://www.centralbank.cw/functions/supervision/supervised-institutions'}
Typology = {regulatorName + ' 1': 'Supervised Institutions'}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

PDF_MAJOR_HEADINGS = {
    'Banks & Non Banks',
    'Insurers & Pension Funds',
    'Securities, Stock Exchange & Trust',

 
}
PDF_MINOR_HEADINGS = {
    'Local General Bank', 'Subsidiary of Foreign Bank', 'Branch of Foreign Bank', 'Credit Union',
    'Specialized Credit Institution', 'Savings Bank', 'Savings and Credit Fund', 'Consolidated International Bank',
    'Non-Consolidated International Bank', 'Appendix I: Coupon Credit article 45',
    'Appendix II: Savings and Thrift Fund article 45',
    'Appendix III: Other institutions or persons in the possession of a dispensation to extend credits directly or indirectly article 45',
    'Money Transfer Company', 'Life Insurance Company', 'Branch of foreign insurance company', 'Independent Company',
    'Subsidiary of foreign insurance company', 'Funeral Services Insurance Company', 'Indemnity Insurance Company',
    'Professional Indemnity Reinsurance Company', 'Captive', 'Indemnity Insurance Captive', 'Life Insurance Captive',
    'Broker', 'Indemnity and Life Insurance Broker', 'Indemnity Insurance Broker', 'Life Insurance Broker',
    'Pension Fund', 'Corporate Pension Fund', 'General Pension Fund', 'Administrator', 'Local Administrator',
    'Asset Management Company', 'Dispensation Asset Manager', 'License Asset Management', 'Investment Institutions',
    'Foreign Investment Fund', 'Securities Exchange', 'Securities Intermediary', 'License Securities Intermediary',
    'Registration Securities Intermediary', 'Trust Service Providers', 'Dispensation Legal Person',
    'Dispensation Natural Person', 'License Legal Person', 'License Natural Person'
}
PDF_GROUP_HEADINGS = {
    'Life Insurance Company', 'Indemnity Insurance Company', 'Captive', 'Broker', 'Pension Fund',
    'Administrator', 'Asset Management Company', 'Investment Institutions', 'Securities Intermediary',
    'Trust Service Providers'
}
PDF_NATURAL_PERSON_SECTIONS = {'Dispensation Natural Person', 'License Natural Person'}
PDF_COUNTRY_CODES = {
    'Curaçao': 'CW', 'Curacao': 'CW', 'Sint Maarten': 'SX', 'United States of America': 'US',
    'Saint Lucia': 'LC', 'Barbados': 'BB', 'Antigua': 'AG', 'Trinidad and Tobago': 'TT',
    'India': 'IN', 'England': 'GB', 'Luxembourg': 'LU'
}
PDF_CITY_DEFAULT_COUNTRY = {
    'Willemstad': 'Curaçao', 'Parrera': 'Curaçao', 'Philipsburg': 'Sint Maarten',
    'Phillipsburg': 'Sint Maarten', 'Philipsbrug': 'Sint Maarten', 'Cole Bay': 'Sint Maarten',
    'Simpson Bay': 'Sint Maarten', 'Cul de Sac': 'Sint Maarten', 'Castries': 'Saint Lucia',
    'Bridgetown': 'Barbados', "St. John's": 'Antigua', 'Port of Spain': 'Trinidad and Tobago',
    'Mumbai': 'India', 'London': 'England', 'Madison': 'United States of America',
    'Florida': 'United States of America', 'St. Michael': 'Barbados'
}
PDF_ADDRESS_WORDS = (
    'box', 'road', 'street', 'straat', 'weg', 'kaya', 'avenue', 'boulevard', 'plein', 'plaza',
    'center', 'centre', 'building', 'complex', 'office', 'park', 'tower', 'mall', 'suite',
    'unit', 'floor', 'landhuis', 'handelskade', 'watersteeg', 'frontstreet', 'corner of', 'c/o'
)
PDF_ENTITY_WORDS = (
    'n.v.', 'b.v.', 's.a.', 'inc', 'limited', 'ltd', 'company', 'bank', 'foundation',
    'stichting', 'fundashon', 'fund', 'insurance', 'trust', 'management', 'services',
    'broker', 'union', 'pension', 'corporation', 'coöperatieve', 'cooperatieve',
    'kooperativa', 'exchange', 'securities', 'capital', 'finance', 'financial', 'asset',
    'group', 'institution', 'reinsurance'
)


def append_sqldict_row(target, row):
    for key in target:
        target[key].append(row.get(key, ''))


def is_pdf_address_line(line):
    low = line.lower()
    return bool(re.search(r'\d', line) or any(word in low for word in PDF_ADDRESS_WORDS))


def is_pdf_person_name(line):
    clean = line.replace('\u200b', '').strip()
    low = clean.lower()
    if low.startswith(('mr.', 'mrs.', 'ms.', 'mw.')):
        return True
    if is_pdf_address_line(clean):
        return False
    has_entity_word = any(word in low for word in PDF_ENTITY_WORDS)
    return bool(re.match(r"^[A-ZÀ-ÿ][^,]{1,60},\s*[A-ZÀ-ÿ]", clean) and not has_entity_word)


def get_pdf_heading(line):
    heading = re.sub(r'\s+', ' ', line.replace('＆', '&')).strip()
    heading = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', heading)
    heading = re.sub(r'\s+\d+(?:\.\d+)?$', '', heading).strip()
    if heading.startswith('Appendix I:'):
        return 'Appendix I: Coupon Credit article 45'
    if heading.startswith('Appendix II:'):
        return 'Appendix II: Savings and Thrift Fund article 45'
    if heading.startswith('Appendix III: Other institutions or persons'):
        return 'Appendix III: Other institutions or persons in the possession of a dispensation to extend credits directly or indirectly article 45'
    if heading.startswith('extend credits directly or indirectly article'):
        return '__skip__'
    if heading in PDF_MAJOR_HEADINGS or heading in PDF_MINOR_HEADINGS:
        return heading
    return ''


def parse_pdf_row(block, current_major, current_group, current_license, reg_parts, list_name):
    lines = [re.sub(r'\s+', ' ', line).strip() for line in block if line.strip()]
    lines = [line for line in lines if not line.lower().startswith(('broker status:', 'actual leader:'))]
    if not lines:
        return None

    websites = [line for line in lines if line.lower().startswith(('http', 'www.'))]
    lines = [line for line in lines if not line.lower().startswith(('http', 'www.'))]
    if not lines:
        return None

    name = re.sub(r'(?<=[A-Za-z)])\d+$', '', lines[0]).strip()
    detail_start = 1
    if detail_start < len(lines) and name.lower().endswith('(in') and lines[detail_start].lower().startswith('liquidation'):
        name = f'{name} {lines[detail_start]}'
        detail_start += 1
    if is_pdf_person_name(name):
        return None

    country_idx = next((idx for idx, value in enumerate(lines) if value in PDF_COUNTRY_CODES), None)
    if country_idx is not None:
        country = lines[country_idx]
        prev_line = lines[country_idx - 1] if country_idx > detail_start else ''
        if prev_line and (not is_pdf_address_line(prev_line) or any(prev_line.startswith(city) for city in PDF_CITY_DEFAULT_COUNTRY)):
            city = prev_line
            detail_end = country_idx - 1
        else:
            city = ''
            detail_end = country_idx
    else:
        last_non_url = lines[-1]
        city = last_non_url if last_non_url in PDF_CITY_DEFAULT_COUNTRY else ''
        country = PDF_CITY_DEFAULT_COUNTRY.get(city, '')
        detail_end = len(lines) - 1 if city else len(lines)

    detail_lines = lines[detail_start:detail_end]
    address_start = next((idx for idx, value in enumerate(detail_lines) if is_pdf_address_line(value)), None)
    if address_start is None:
        address_start = 0 if detail_lines else len(detail_lines)

    aliases = [value.strip('[]') for value in detail_lines[:address_start] if value.strip('[]')]
    address_lines = detail_lines[address_start:]
    po_boxes = [line for line in address_lines if line.lower().startswith(('po box', 'p.o. box'))]
    street_lines = [line for line in address_lines if line not in po_boxes]
    license_type = current_license or current_group
    if current_group and current_license and current_group != current_license:
        license_type = f'{current_group} - {current_license}'

    RegCtry, RegCode, ListCode = reg_parts
    return {
        'Name': name,
        'Name - Mother Company': '; '.join(aliases),
        'Address_1': ', '.join(street_lines),
        'Address_2': '; '.join(po_boxes),
        'City': city,
        'Cntry': PDF_COUNTRY_CODES.get(country, country),
        'Website': '; '.join(dict.fromkeys(websites)),
        'CoType': current_major,
        'License_Type': license_type,
        'RegulationType': 'Regulated',
        'RegCtry': RegCtry,
        'RegCode': RegCode,
        'ListCode': ListCode,
        'ListName': list_name,
        'ListProcessDate': processdate,
    }


def extract_cbcscw_pdf_rows(pdf_path, reg_parts, list_name):
    rows, unparsed_blocks = [], []
    current_major, current_group, current_license = '', '', ''
    natural_section = False
    skip_exhibit_legal = False
    skip_detail_until_country = False
    block = []

    def finish_block():
        nonlocal block
        if not block:
            return
        row = parse_pdf_row(block, current_major, current_group, current_license, reg_parts, list_name)
        if row:
            rows.append(row)
        elif len(block) > 1:
            unparsed_blocks.append(block.copy())
        block = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num <= 3:
                continue
            skip_footnote = False
            text = page.extract_text() or ''
            for raw_line in text.splitlines():
                line = re.sub(r'\s+', ' ', raw_line.replace('\xa0', ' ')).strip()
                if not line:
                    continue
                if re.fullmatch(r'--\s*\d+\s+of\s+\d+\s*--', line) or re.fullmatch(r'-\s*\d+\s*-', line):
                    skip_footnote = False
                    continue
                if skip_footnote:
                    continue
                if re.match(r'^\d+\s+The\s+', line):
                    skip_footnote = True
                    continue

                heading = get_pdf_heading(line)
                if heading == '__skip__':
                    continue
                if heading:
                    finish_block()
                    if heading in PDF_MAJOR_HEADINGS:
                        current_major, current_group, current_license = heading, '', ''
                        natural_section = False
                    else:
                        natural_section = heading in PDF_NATURAL_PERSON_SECTIONS
                        if heading in PDF_GROUP_HEADINGS:
                            current_group, current_license = heading, ''
                        else:
                            current_license = heading
                    skip_exhibit_legal = False
                    continue

                low = line.lower()
                if skip_detail_until_country:
                    if line in PDF_COUNTRY_CODES:
                        skip_detail_until_country = False
                    continue
                if natural_section or low.startswith(('indemnity group', 'a:', 'b:', 'c:', 'd:', 'e:', 'f:')):
                    continue
                if low.startswith(("exhibit 'a'", 'exhibit a')):
                    finish_block()
                    skip_exhibit_legal = True
                    continue
                if low.startswith(("exhibit 'b'", 'exhibit b', 'exhbit b')):
                    finish_block()
                    skip_exhibit_legal = False
                    continue
                if low.startswith('local representative:'):
                    finish_block()
                    skip_detail_until_country = True
                    continue
                if skip_exhibit_legal:
                    continue
                if is_pdf_person_name(line):
                    if not low.startswith(('mr.', 'mrs.', 'ms.', 'mw.')):
                        skip_detail_until_country = True
                    continue
                if low.startswith(('actual leader:', 'broker status:')):
                    continue
                if not block and (line in PDF_COUNTRY_CODES or line in PDF_CITY_DEFAULT_COUNTRY):
                    continue
                if line.lower().startswith(('http', 'www.')):
                    if block:
                        block.append(line)
                    elif rows:
                        rows[-1]['Website'] = '; '.join(filter(None, [rows[-1].get('Website', ''), line]))
                    continue

                last_context = next((value for value in reversed(block) if not value.lower().startswith(('http', 'www.'))), '')
                if block and last_context in PDF_CITY_DEFAULT_COUNTRY and line not in PDF_COUNTRY_CODES and not is_pdf_address_line(line):
                    finish_block()
                block.append(line)
                if line in PDF_COUNTRY_CODES:
                    finish_block()

    finish_block()
    return rows, unparsed_blocks


# Bold-font parser: entity names in the PDF are visually marked in bold black text.
def is_pdf_black_color(color):
    if color is None:
        return True
    if isinstance(color, (int, float)):
        return color == 0
    if isinstance(color, (list, tuple)):
        return all(value in (0, 0.0) for value in color)
    return False


def is_pdf_bold_font(fontname):
    font = str(fontname or '').lower()
    return any(token in font for token in ('bold', 'black', 'heavy', 'bd'))


def clean_pdf_text(text):
    return re.sub(r'\s+', ' ', str(text).replace('\xa0', ' ')).strip()


def extract_pdf_visual_lines(page):
    chars = [char for char in page.chars if clean_pdf_text(char.get('text', ''))]
    if not chars:
        return []

    lines = []
    for char in sorted(chars, key=lambda item: (round(item.get('top', 0), 1), item.get('x0', 0))):
        if not lines or abs(char.get('top', 0) - lines[-1]['top']) > 2.5:
            lines.append({'top': char.get('top', 0), 'chars': [char]})
        else:
            lines[-1]['chars'].append(char)

    visual_lines = []
    for line in lines:
        line_chars = sorted(line['chars'], key=lambda item: item.get('x0', 0))
        parts = []
        previous_x1 = None
        previous_size = 8
        for char in line_chars:
            text = char.get('text', '')
            if previous_x1 is not None and char.get('x0', 0) - previous_x1 > max(1.5, previous_size * 0.25):
                parts.append(' ')
            parts.append(text)
            previous_x1 = char.get('x1', char.get('x0', 0))
            previous_size = char.get('size', previous_size)

        text = clean_pdf_text(''.join(parts))
        if not text:
            continue

        non_space_chars = [char for char in line_chars if clean_pdf_text(char.get('text', ''))]
        bold_count = sum(is_pdf_bold_font(char.get('fontname')) for char in non_space_chars)
        black_count = sum(is_pdf_black_color(char.get('non_stroking_color')) for char in non_space_chars)
        sizes = sorted(char.get('size', 0) for char in non_space_chars)
        size = sizes[len(sizes) // 2] if sizes else 0
        visual_lines.append({
            'text': text,
            'bold_ratio': bold_count / len(non_space_chars),
            'black_ratio': black_count / len(non_space_chars),
            'size': size,
            'top': line['top'],
            'x0': min(char.get('x0', 0) for char in line_chars),
        })

    for idx, line in enumerate(visual_lines):
        line['gap_before'] = 999 if idx == 0 else line['top'] - visual_lines[idx - 1]['top']

    return visual_lines


def is_pdf_metadata_line(text):
    low = text.lower()
    return bool(
        re.fullmatch(r'--\s*\d+\s+of\s+\d+\s*--', text)
        or re.fullmatch(r'-\s*\d+\s*-', text)
        or low.startswith(('registry of supervised institutions', 'as per ', 'table of contents'))
        or re.fullmatch(r'\d+', text)
    )


def is_pdf_bold_entity_line(line, natural_section, starts_after_heading=False):
    text = line['text']
    low = text.lower()
    if natural_section or is_pdf_metadata_line(text) or get_pdf_heading(text):
        return False
    if low.startswith(('http', 'www.', 'exhibit ', "exhibit '", 'actual leader:', 'broker status:', 'local representative:', 'indemnity group')):
        return False
    if text in PDF_COUNTRY_CODES or text in PDF_CITY_DEFAULT_COUNTRY or re.match(r'^[a-f]:\s', low):
        return False

    # The PDF does not expose a Bold font name; entity names are same black font style,
    # but they start a new visual block with extra vertical spacing.
    style_match = line['black_ratio'] >= 0.75 and 8.5 <= line['size'] <= 11.5 and line.get('x0', 999) <= 90
    block_start = starts_after_heading or line.get('gap_before', 0) >= 18
    return style_match and block_start and len(text) > 1


def parse_pdf_row(block, current_major, current_group, current_license, reg_parts, list_name):
    lines = [clean_pdf_text(line) for line in block if clean_pdf_text(line)]
    if not lines:
        return None

    notes = [line.removeprefix('Note:').strip() for line in lines if line.startswith('Note:')]
    lines = [line for line in lines if not line.startswith('Note:')]
    websites = [line for line in lines if line.lower().startswith(('http', 'www.'))]
    lines = [line for line in lines if not line.lower().startswith(('http', 'www.'))]
    if not lines:
        return None

    name = re.sub(r'(?<=[A-Za-z)])\d+$', '', lines[0]).strip()
    detail_start = 1
    while detail_start < len(lines) and name.count('(') > name.count(')') and detail_start <= 2:
        name = f'{name} {lines[detail_start]}'
        detail_start += 1

    country_idx = next((idx for idx, value in enumerate(lines) if value in PDF_COUNTRY_CODES), None)
    if country_idx is not None:
        country = lines[country_idx]
        prev_line = lines[country_idx - 1] if country_idx > detail_start else ''
        if prev_line and (not is_pdf_address_line(prev_line) or prev_line in PDF_CITY_DEFAULT_COUNTRY):
            city = prev_line
            detail_end = country_idx - 1
        else:
            city = ''
            detail_end = country_idx
    else:
        city = lines[-1] if lines[-1] in PDF_CITY_DEFAULT_COUNTRY else ''
        country = PDF_CITY_DEFAULT_COUNTRY.get(city, '')
        detail_end = len(lines) - 1 if city else len(lines)

    detail_lines = lines[detail_start:detail_end]
    address_start = next((idx for idx, value in enumerate(detail_lines) if is_pdf_address_line(value)), None)
    if address_start is None:
        address_start = 0 if detail_lines else len(detail_lines)

    aliases = [value.strip('[]') for value in detail_lines[:address_start] if value.strip('[]')]
    address_lines = detail_lines[address_start:]
    po_boxes = [line for line in address_lines if line.lower().startswith(('po box', 'p.o. box'))]
    street_lines = [line for line in address_lines if line not in po_boxes]
    license_type = current_license or current_group
    if current_group and current_license and current_group != current_license:
        license_type = f'{current_group} - {current_license}'

    RegCtry, RegCode, ListCode = reg_parts
    return {
        'Name': name,
        'Name - Mother Company': '; '.join(aliases),
        'Address_1': ', '.join(street_lines),
        'Address_2': '; '.join(po_boxes),
        'City': city,
        'Cntry': PDF_COUNTRY_CODES.get(country, country),
        'Website': '; '.join(dict.fromkeys(websites)),
        'CoType': current_major,
        'License_Type': license_type,
        'RegulationType': 'Regulated',
        'RegCtry': RegCtry,
        'RegCode': RegCode,
        'ListCode': ListCode,
        'ListName': list_name,
        'ListProcessDate': processdate,
        'Check': '; '.join(notes),
    }


def extract_cbcscw_pdf_rows(pdf_path, reg_parts, list_name):
    rows, unparsed_blocks = [], []
    current_major, current_group, current_license = '', '', ''
    natural_section = False
    skip_detail = False
    starts_after_heading = False
    block = []

    def finish_block():
        nonlocal block
        if not block:
            return
        row = parse_pdf_row(block, current_major, current_group, current_license, reg_parts, list_name)
        if row:
            rows.append(row)
        elif len(block) > 1:
            unparsed_blocks.append(block.copy())
        block = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num <= 3:
                continue
            skip_footnote = False
            for line in extract_pdf_visual_lines(page):
                text = clean_pdf_text(line['text'])
                if not text or is_pdf_metadata_line(text):
                    skip_footnote = False
                    continue
                if skip_footnote:
                    continue
                if re.match(r'^\d+\s+The\s+', text):
                    skip_footnote = True
                    continue

                heading = get_pdf_heading(text)
                if heading == '__skip__':
                    continue
                if heading:
                    finish_block()
                    skip_detail = False
                    starts_after_heading = True
                    if heading in PDF_MAJOR_HEADINGS:
                        current_major, current_group, current_license = heading, '', ''
                        natural_section = False
                    else:
                        natural_section = heading in PDF_NATURAL_PERSON_SECTIONS
                        if heading in PDF_GROUP_HEADINGS:
                            current_group, current_license = heading, ''
                        else:
                            current_license = heading
                    continue

                if is_pdf_bold_entity_line(line, natural_section, starts_after_heading):
                    finish_block()
                    block = [text]
                    skip_detail = False
                    starts_after_heading = False
                    continue

                if natural_section or not block:
                    continue

                starts_after_heading = False
                low = text.lower()
                if low.startswith(('local representative:', "exhibit 'a'", 'exhibit a', "exhibit 'b'", 'exhibit b', 'exhbit b')):
                    block.append(f'Note: {text}')
                    skip_detail = True
                    continue
                if low.startswith(('actual leader:', 'broker status:', 'indemnity group')) or re.match(r'^[a-f]:\s', low):
                    block.append(f'Note: {text}')
                    continue
                if skip_detail:
                    block.append(f'Note: {text}')
                    if text in PDF_COUNTRY_CODES:
                        skip_detail = False
                    continue
                block.append(text)

    finish_block()
    return rows, unparsed_blocks


In [16]:
for k, (reg, base_url) in enumerate(regdict.items()):
    print(f'[INFO] : Working {k+1}/{len(regdict)} _({reg})_')
    parts = reg.split()
    RegCtry, RegCode, ListCode = parts[0], parts[1], parts[2]

    if registry_pdf_files:
        for pdf_path in registry_pdf_files:
            before_count = len(sqldict['Name'])
            print(f'[INFO] : extracting registry PDF {os.path.basename(pdf_path)}')
            pdf_rows, unparsed_blocks = extract_cbcscw_pdf_rows(pdf_path, (RegCtry, RegCode, ListCode), Typology[reg])
            for row in pdf_rows:
                append_sqldict_row(sqldict, row)

            print(f'[INFO] : PDF rows added {len(sqldict["Name"]) - before_count}; total rows {len(sqldict["Name"])}')
            if pdf_rows:
                pdf_preview = pd.DataFrame(pdf_rows)
                preview_cols = ['Name', 'License_Type', 'Address_1', 'Address_2', 'City', 'Cntry', 'Website']
                print(pdf_preview[preview_cols].head(10).to_string(index=False))

                expected_names = ['Chenurrai N.V.', 'ENCA N.V.', 'Admico N.V.']
                missing_names = [name for name in expected_names if not pdf_preview['Name'].eq(name).any()]
                if missing_names:
                    print(f'[WARN] : expected PDF names missing: {missing_names}')
                else:
                    print(f'[INFO] : expected PDF names found: {expected_names}')
            if unparsed_blocks:
                print(f'[WARN] : {len(unparsed_blocks)} PDF block(s) were not parsed; first examples:')
                for block in unparsed_blocks[:5]:
                    print('  - ' + ' | '.join(block[:5]))
        continue

    driver.get(base_url)
    sleep(30)
    if 'just a moment' in driver.title.lower():
        raise RuntimeError('Cloudflare challenge not bypassed; rerun the cell to retry.')

    soup = BeautifulSoup(driver.html, 'html.parser')
    page_links = [a for a in soup.select('a.page-link') if a.get_text(strip=True).isdigit()]
    last_page = max((int(a.get_text(strip=True)) for a in page_links), default=1)
    print(f'[INFO] : detected {last_page} pages')

    # for page_num in range(1, last_page + 1):
    for page_num in range(1, 2):
        if page_num > 1:
            driver.get(f'{base_url}?page={page_num}&query=&category=&options=')
            sleep(5)
            soup = BeautifulSoup(driver.html, 'html.parser')

        items = soup.select('li.py-4')
        if not items:
            raise RuntimeError(f'No li.py-4 entities on page {page_num}; layout may have changed.')

        for li in items:
            heading = li.find('h4')
            body = li.find('div', id=lambda x: x and x.startswith('entry-'))
            if heading is None or body is None:
                continue

            name = heading.get_text(' ', strip=True)
            strong_tag = body.find('strong')
            category = strong_tag.get_text(' ', strip=True) if strong_tag else ''

            paragraphs = []
            for ptag in body.find_all('p'):
                for line in ptag.get_text('\n').splitlines():
                    line = line.strip().replace('\xa0', ' ')
                    if line:
                        paragraphs.append(line)

            license_type, trade_name, po_box = '', '', ''
            phone, fax, email, website = '', '', '', ''
            address_lines = []
            license_assigned = False

            for line in paragraphs:
                low = line.lower()
                if line.startswith('Trade name:'):
                    trade_name = line.split(':', 1)[1].strip()
                elif category and line == category:
                    continue
                elif low.startswith('phone'):
                    phone = line.split(':', 1)[1].strip() if ':' in line else line
                elif low.startswith('fax'):
                    fax = line.split(':', 1)[1].strip() if ':' in line else line
                elif '@' in line and ' ' not in line:
                    email = line
                elif low.startswith('http') or low.startswith('www.'):
                    website = line
                elif low.startswith('p.o. box') or low.startswith('po box'):
                    po_box = line.split(':', 1)[1].strip() if ':' in line else line
                elif low.startswith("exhibit '") or low.startswith('actual leader') or low.startswith('mr.') or low.startswith('mrs.'):
                    continue
                elif not license_assigned and category:
                    license_type = line
                    license_assigned = True
                else:
                    address_lines.append(line)

            address_1 = ', '.join(address_lines[:-1]) if len(address_lines) > 1 else (address_lines[0] if address_lines else '')
            city_country = address_lines[-1] if len(address_lines) > 1 else ''

            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Cntry'].append('CW')
            sqldict['RegCtry'].append(RegCtry)
            sqldict['RegCode'].append(RegCode)
            sqldict['ListCode'].append(ListCode)
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict['CoType'].append(category)
            sqldict['License_Type'].append(license_type)
            sqldict['Address_1'].append(address_1)
            sqldict['Address_2'].append(po_box)
            sqldict['City'].append(city_country)
            sqldict['Phone'].append(phone)
            sqldict['Fax'].append(fax)
            sqldict['Email'].append(email)
            sqldict['Website'].append(website)
            sqldict['Name - Mother Company'].append(trade_name)

            row_len = len(sqldict['Name'])
            for key in sqldict:
                if len(sqldict[key]) < row_len:
                    sqldict[key].append('')

        print(f'[INFO] : page {page_num} -> total rows {len(sqldict["Name"])}')

if not sqldict['Name']:
    raise RuntimeError('No CW CBCSCW rows were scraped or parsed.')


[INFO] : Working 1/1 _(CW CBCSCW 1)_
[INFO] : extracting registry PDF 20260211_registry_of_supervised_institutions_as_per_december_31_2025.pdf
[INFO] : PDF rows added 345; total rows 345
                               Name               License_Type                                               Address_1   Address_2         City Cntry                                Website
                      APC Bank N.V.         Local General Bank                                        De Ruyterkade 61               Willemstad    CW                 https://www.apcbank.cw
               APC Bank St. Maarten         Local General Bank Miss Lalies Commercial Centre, Bush Road 26, Unit 1 & 2              Philipsburg    SX                                       
               Banco di Caribe N.V.         Local General Bank                                  Schottegatweg Oost 205 PO Box 3785   Willemstad    CW          https://www.bancodicaribe.com
Banco di Caribe N.V. (Sint Maarten)         Local General

In [8]:
field_lengths = {key: len(value) for key, value in sqldict.items()}
if len(set(field_lengths.values())) != 1:
    raise RuntimeError(f'SQL output columns have uneven lengths: {field_lengths}')

df = pd.DataFrame(sqldict)
if df.empty:
    raise RuntimeError('No rows available for export.')

sample_cols = ['Name', 'License_Type', 'Address_1', 'Address_2', 'City', 'Cntry', 'Website']
print('[INFO] : output sample')
print(df[sample_cols].head(10).to_string(index=False))

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)
print(f'[INFO] : wrote {len(df)} rows to {filename}')

if driver is not None:
    driver.quit()
sleep(3)

if cleanup_tempfolder_after_export:
    temp_path = os.path.abspath(tempfolder)
    if os.path.basename(temp_path).lower() == 'tempfolder' and os.path.dirname(temp_path) == os.path.abspath(scriptfolder):
        for rem in os.listdir(tempfolder):
            rem_path = os.path.join(tempfolder, rem)
            if os.path.isfile(rem_path):
                os.remove(rem_path)
    else:
        raise RuntimeError(f'Unsafe tempfolder cleanup path: {tempfolder}')
else:
    print('[INFO] : tempfolder cleanup skipped; set cleanup_tempfolder_after_export=True after validation')


[INFO] : output sample
                                                      Name                        License_Type                                                                                                                                    Address_1 Address_2                      City Cntry                 Website
(SMES) Solutions for Management and Employment Support N.V                License Legal Person                                                                                                                      Dr. Henri Fergusonweg 1                 Willemstad, Curaçao    CW                        
                                               Admico N.V.                License Legal Person                                                                                                                              Cas Coraweg 115                 Willemstad, Curaçao    CW                        
                                                   Akeem's Indemnity an

In [19]:
registry_pdf_files

["C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\CW CBCSCW\\tempfolder\\20260211_registry_of_supervised_institutions_as_per_december_31_2025.pdf"]

In [ ]:
import re
import pdfplumber

records = []
current_record = None
pdf_headings = set().union(
    globals().get('PDF_MAJOR_HEADINGS', set()),
    globals().get('PDF_MINOR_HEADINGS', set()),
    globals().get('PDF_GROUP_HEADINGS', set()),
)


def clean_chars(line_chars):
    line_chars = sorted(line_chars, key=lambda item: item.get('x0', 0))
    parts = []
    previous_x1 = None
    previous_size = 9

    for char in line_chars:
        if previous_x1 is not None and char.get('x0', 0) - previous_x1 > max(1.5, previous_size * 0.25):
            parts.append(' ')
        parts.append(char.get('text', ''))
        previous_x1 = char.get('x1', char.get('x0', 0))
        previous_size = char.get('size', previous_size)

    return re.sub(r'\s+', ' ', ''.join(parts)).strip()


with pdfplumber.open(registry_pdf_files[0]) as pdf:
    for page in pdf.pages:
        page_lines = {}
        for char in page.chars:
            if char.get('object_type') != 'char' or 'BKSWEM+MuseoSans-300' not in char.get('fontname', ''):
                continue
            if char.get('size', 0) <= 9:
                continue
            page_lines.setdefault(round(char.get('top', 0), 1), []).append(char)

        for top in sorted(page_lines):
            line_chars = page_lines[top]
            name_text = clean_chars([char for char in line_chars if char.get('stroking_color') == (0.0,)])
            info_text = clean_chars([char for char in line_chars if char.get('stroking_color') == (1.0,)])

            if info_text and re.sub(r'\s+\d+(?:\.\d+)?$', '', info_text).strip() in pdf_headings:
                info_text = ''

            if name_text:
                current_record = {'name': name_text, 'info': []}
                records.append(current_record)

            if info_text and current_record is not None:
                current_record['info'].append(info_text)

name = []
info_blocks = []

for record in records:
    clean_name = re.sub(r'\s+', ' ', record['name']).strip()
    if not clean_name or clean_name.isdigit() or not re.search(r'[A-Za-zÀ-ÿ]', clean_name):
        continue

    info_block = re.sub(r'\s+', ' ', ' '.join(record['info'])).strip()

    # Remove bracket notes like [Sint Maarten branch]
    info_block = re.sub(r'\[[^\]]*\]\s*', '', info_block)

    # Remove Exhibit A/B and everything after
    info_block = re.sub(r'\bExhibit\s+[AB]\b.*$', '', info_block)

    # Remove Broker status and Actual leader info
    info_block = re.sub(r'\bBroker status:\s*.*?(?=\bActual leader:|$)', '', info_block)
    info_block = re.sub(r'\bActual leader:\s*.*$', '', info_block)

    info_block = re.sub(r'\s+', ' ', info_block).strip()

    name.append(clean_name)
    info_blocks.append(info_block)
    
info_blocks = [
    re.sub(r'\bExhibit\s+[AB]\b.*$', '', re.sub(r'\[[^\]]*\]\s*', '', item)).strip()
    for item in info_blocks
]

In [145]:
for na, info in list(zip(name, info_blocks)):
    print('Name: ->',na)
    index_web = info.find('http')
    if index_web != -1:
        print(info[index_web:])
        
        clean_text = re.sub(r'\[[^\]]*\]\s*', '', info[:index_web]).strip()
        print(clean_text)
    else:
        
        clean_text = re.sub(r'\[[^\]]*\]\s*', '', info).strip()
        print(clean_text)

Name: -> APC Bank N.V.
https://www.apcbank.cw
De Ruyterkade 61 Willemstad Curaçao
Name: -> APC Bank St. Maarten
Miss Lalies Commercial Centre Bush Road 26, Unit 1 & 2 Philipsburg Sint Maarten
Name: -> Banco di Caribe N.V.
https://www.bancodicaribe.com
Schottegatweg Oost 205 PO Box 3785 Willemstad Curaçao
Name: -> Banco di Caribe N.V. (Sint Maarten)
https://www.bancodicaribe.com
Laguna View Professional Center, Welfare Road 44 Cole Bay Sint Maarten
Name: -> Maduro & Curiel's Bank N.V.
https://www.mcb-bank.com
Plaza Jojo Correa 2-4 PO Box 305 Willemstad Curaçao
Name: -> Orco Bank N.V.
https://www.orcobank.com
Landhuis Cerrito Schottegatweg Oost z/n PO Box 4928 Willemstad Curaçao
Name: -> Orco Bank N.V. (Branch Union Plaza)
https://www.orcobank.com
Emmaplein 1 Philipsburg Sint Maarten
Name: -> The Windward Islands Bank

Name: -> MCB Sint Maarten Branch
http://www.wib-bank.net
Cannegieter street Phillipsburg Sint Maarten
Name: -> Vidanova Bank N.V.
https://www.vidanovabank.com
Schottegatwe